In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

import warnings
warnings.filterwarnings("ignore")

In [2]:
train_df = pd.read_csv("../data/processed/clean_train.csv")
test_df = pd.read_csv("../data/processed/clean_test.csv")

In [3]:
train_df.head()

,date,store,item,sales,year,month,day,day_of_week,week
0,2013-01-01,1,1,13,2013,1,1,1,1
1,2013-01-01,7,12,26,2013,1,1,1,1
2,2013-01-01,7,46,27,2013,1,1,1,1
3,2013-01-01,8,12,54,2013,1,1,1,1
4,2013-01-01,9,12,35,2013,1,1,1,1


In [4]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   date         913000 non-null  str  
 1   store        913000 non-null  int64
 2   item         913000 non-null  int64
 3   sales        913000 non-null  int64
 4   year         913000 non-null  int64
 5   month        913000 non-null  int64
 6   day          913000 non-null  int64
 7   day_of_week  913000 non-null  int64
 8   week         913000 non-null  int64
dtypes: int64(8), str(1)
memory usage: 62.7 MB


In [6]:
train_df["date"] = pd.to_datetime(train_df["date"])

In [7]:
train_df["year"] = train_df["date"].dt.year
train_df["month"] = train_df["date"].dt.month
train_df["day"] = train_df["date"].dt.day
train_df["day_of_week"] = train_df["date"].dt.dayofweek
train_df["week"] = train_df["date"].dt.isocalendar().week.astype(int)

In [8]:
train_df["quarter"] = train_df["date"].dt.quarter

train_df["is_weekend"] = (
    train_df["day_of_week"]
    .isin([5, 6])
    .astype(int)
)

In [9]:
train_df.head()

,date,store,item,sales,year,month,day,day_of_week,week,quarter,is_weekend
0,2013-01-01,1,1,13,2013,1,1,1,1,1,0
1,2013-01-01,7,12,26,2013,1,1,1,1,1,0
2,2013-01-01,7,46,27,2013,1,1,1,1,1,0
3,2013-01-01,8,12,54,2013,1,1,1,1,1,0
4,2013-01-01,9,12,35,2013,1,1,1,1,1,0


In [10]:
# Create Lag Features
# Lag features represent previous sales values.
# Previous day's sales
train_df["lag_1"] = train_df["sales"].shift(1)

# Previous week's sales
train_df["lag_7"] = train_df["sales"].shift(7)

# Previous month's sales
train_df["lag_30"] = train_df["sales"].shift(30)

In [11]:
train_df[["sales","lag_1","lag_7","lag_30"]].head(35)

,sales,lag_1,lag_7,lag_30
0,13,NaN,NaN,NaN
1,26,13.0,NaN,NaN
2,27,26.0,NaN,NaN
3,54,27.0,NaN,NaN
4,35,54.0,NaN,NaN
5,41,35.0,NaN,NaN
6,23,41.0,NaN,NaN
7,37,23.0,13.0,NaN
8,51,37.0,26.0,NaN
9,20,51.0,27.0,NaN


In [12]:
train_df["rolling_mean_7"] = (
    train_df["sales"]
    .rolling(window=7)
    .mean()
)

train_df["rolling_mean_30"] = (
    train_df["sales"]
    .rolling(window=30)
    .mean()
)

In [13]:
train_df[
    [
        "sales",
        "rolling_mean_7",
        "rolling_mean_30"
    ]
].head(40)

,sales,rolling_mean_7,rolling_mean_30
0,13,NaN,NaN
1,26,NaN,NaN
2,27,NaN,NaN
3,54,NaN,NaN
4,35,NaN,NaN
5,41,NaN,NaN
6,23,31.285714,NaN
7,37,34.714286,NaN
8,51,38.285714,NaN
9,20,37.285714,NaN


In [14]:
train_df["rolling_std_7"] = (
    train_df["sales"]
    .rolling(window=7)
    .std()
)

train_df["rolling_std_30"] = (
    train_df["sales"]
    .rolling(window=30)
    .std()
)

In [15]:
train_df["expanding_mean"] = (
    train_df["sales"]
    .expanding()
    .mean()
)

In [16]:
train_df.isnull().sum()

date                0
store               0
item                0
sales               0
year                0
month               0
day                 0
day_of_week         0
week                0
quarter             0
is_weekend          0
lag_1               1
lag_7               7
lag_30             30
rolling_mean_7      6
rolling_mean_30    29
rolling_std_7       6
rolling_std_30     29
expanding_mean      0
dtype: int64

In [17]:
train_df = train_df.dropna().reset_index(drop=True)

In [18]:
train_df.head()

,date,store,item,sales,year,month,day,day_of_week,week,quarter,is_weekend,lag_1,lag_7,lag_30,rolling_mean_7,rolling_mean_30,rolling_std_7,rolling_std_30,expanding_mean
0,2013-01-01,5,12,22,2013,1,1,1,1,1,0,28.0,37.0,13.0,31.285714,36.500000,9.268482,10.254520,35.741935
1,2013-01-01,4,12,36,2013,1,1,1,1,1,0,22.0,30.0,26.0,32.142857,36.833333,9.406178,10.062163,35.750000
2,2013-01-01,4,10,30,2013,1,1,1,1,1,0,36.0,43.0,27.0,30.285714,36.933333,8.097619,9.975602,35.575758
3,2013-01-01,5,10,31,2013,1,1,1,1,1,0,30.0,22.0,54.0,31.571429,36.166667,7.230886,9.490770,35.441176
4,2013-01-01,6,10,29,2013,1,1,1,1,1,0,31.0,45.0,35.0,29.285714,35.966667,4.151879,9.579012,35.257143


In [19]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 912970 entries, 0 to 912969
Data columns (total 19 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   date             912970 non-null  datetime64[us]
 1   store            912970 non-null  int64         
 2   item             912970 non-null  int64         
 3   sales            912970 non-null  int64         
 4   year             912970 non-null  int32         
 5   month            912970 non-null  int32         
 6   day              912970 non-null  int32         
 7   day_of_week      912970 non-null  int32         
 8   week             912970 non-null  int64         
 9   quarter          912970 non-null  int32         
 10  is_weekend       912970 non-null  int64         
 11  lag_1            912970 non-null  float64       
 12  lag_7            912970 non-null  float64       
 13  lag_30           912970 non-null  float64       
 14  rolling_mean_7   912970 non-nul

In [20]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv(
    "../data/processed/feature_engineered_train.csv",
    index=False
)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.


In [21]:
features = [
    "store",
    "item",
    "year",
    "month",
    "day",
    "day_of_week",
    "week",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_mean_7",
    "rolling_mean_30",
    "rolling_std_7",
    "rolling_std_30",
    "expanding_mean"
]

target = "sales"

In [22]:
X = train_df[features]

y = train_df[target]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Shape : (912970, 17)
Target Shape : (912970,)


In [23]:
from sklearn.preprocessing import MinMaxScaler

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_scaled = feature_scaler.fit_transform(X)

y_scaled = target_scaler.fit_transform(
    y.values.reshape(-1,1)
)

In [24]:
print(X_scaled.shape)
print(y_scaled.shape)

(912970, 17)
(912970, 1)


In [25]:
def create_sequences(X, y, sequence_length=30):

    X_seq = []
    y_seq = []

    for i in range(len(X)-sequence_length):

        X_seq.append(X[i:i+sequence_length])

        y_seq.append(y[i+sequence_length])

    return np.array(X_seq), np.array(y_seq)

In [27]:
X_scaled = X_scaled.astype(np.float32)
y_scaled = y_scaled.astype(np.float32)

In [29]:
sequence_length = 15

X_seq, y_seq = create_sequences(
    X_scaled,
    y_scaled,
    sequence_length
)

In [30]:
print("Sequence Shape :", X_seq.shape)

print("Target Shape :", y_seq.shape)

Sequence Shape : (912955, 15, 17)
Target Shape : (912955, 1)


In [31]:
split_index = int(len(X_seq) * 0.8)

X_train = X_seq[:split_index]
X_test = X_seq[split_index:]

y_train = y_seq[:split_index]
y_test = y_seq[split_index:]

In [32]:
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

X_train : (730364, 15, 17)
X_test : (182591, 15, 17)
y_train : (730364, 1)
y_test : (182591, 1)


In [33]:
train_df["lag_1"] = (
    train_df.groupby(["store", "item"])["sales"]
            .shift(1)
)

In [34]:
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    RepeatVector,
    TimeDistributed
)

from tensorflow.keras.models import Model

In [35]:
sequence_length = X_train.shape[1]

n_features = X_train.shape[2]

latent_dim = 64

In [36]:
encoder_inputs = Input(
    shape=(sequence_length, n_features)
)

encoder = LSTM(
    latent_dim,
    activation="tanh",
    return_state=True
)

encoder_outputs, state_h, state_c = encoder(
    encoder_inputs
)

encoder_states = [state_h, state_c]

In [37]:
decoder_inputs = RepeatVector(1)(encoder_outputs)

In [38]:
decoder_lstm = LSTM(
    latent_dim,
    activation="tanh",
    return_sequences=True
)

decoder_outputs = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

In [40]:
outputs = TimeDistributed(
    Dense(1)
)(decoder_outputs)

In [41]:
model = Model(
    encoder_inputs,
    outputs
)

In [42]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 15, 17)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 64),      │     20,992 │ input_layer[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 1, 64)     │          0 │ lstm[0][0]        │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 1, 64)     │     33,024 │ repeat_vector[0]… │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 1, 1)      │         65 │ lstm_2[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 54,081 (211.25 KB)

 Trainable params: 54,081 (211.25 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [44]:
history = model.fit(
    X_train,
    y_train.reshape(-1,1,1),
    validation_data=(
        X_test,
        y_test.reshape(-1,1,1)
    ),
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
11412/11412 ━━━━━━━━━━━━━━━━━━━━ 145s 12ms/step - loss: 0.0120 - mae: 0.0888 - val_loss: 0.0157 - val_mae: 0.1015
Epoch 2/20
11412/11412 ━━━━━━━━━━━━━━━━━━━━ 136s 12ms/step - loss: 0.0119 - mae: 0.0884 - val_loss: 0.0157 - val_mae: 0.1021
Epoch 3/20
 5125/11412 ━━━━━━━━━━━━━━━━━━━━ 1:51 18ms/step - loss: 0.0119 - mae: 0.0883

: 